# 04 — Ablation Results

Compare the three agent conditions across all evaluation metrics:

| Condition | Agent Mode | Compressor |
|-----------|-----------|------------|
| Baseline 1 | `raw` | none |
| Baseline 2 | `llm_summary` | LLMCompressor |
| Trained | `compressor` | TransformerCompressor (PPO) |

Loads eval result JSON files from `outputs/eval_results/`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
from pathlib import Path

from optimized_llm_planning_memory.core.models import EvalResult

eval_dir = Path('../outputs/eval_results')
result_files = list(eval_dir.glob('*.json')) if eval_dir.exists() else []
print(f'Found {len(result_files)} eval result files')

In [ ]:
# Load and aggregate
from optimized_llm_planning_memory.evaluation.evaluator import Evaluator

all_results = {}
for f in result_files:
    raw = json.loads(f.read_text())
    results = [EvalResult.model_validate(r) for r in raw]
    evaluator = Evaluator()
    agg = evaluator.aggregate(results)
    all_results[f.stem] = agg
    print(f'\n=== {f.stem} ===')
    for k, v in sorted(agg.items()):
        if k.endswith('_mean'):
            print(f'  {k:<45} {v:.4f}')

In [ ]:
# Side-by-side comparison table
key_metrics = [
    'hard_constraint_ratio_mean',
    'soft_constraint_score_mean',
    'tool_efficiency_mean',
    'budget_adherence_mean',
    'logical_consistency_mean',
    'overall_score_mean',
]

header = f'{"Run":<40}' + ''.join(f"{m.replace("_mean",""):>20}" for m in key_metrics)
print(header)
print('-' * len(header))
for run_name, agg in sorted(all_results.items()):
    row = f'{run_name:<40}'
    for m in key_metrics:
        val = agg.get(m, float('nan'))
        row += f'{val:>20.4f}'
    print(row)